# EfficientNet-B7 Hair Type Classifier

This notebook trains a hair type classifier using EfficientNet-B7 with mixed precision training.

## Setup Instructions
1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `GPU` (preferably T4 or better)
2. **Data structure**: Uses `data/segmented/` with class folders:
```
data/
└── segmented/
    ├── 1/
    ├── 1a/
    ├── 2a/
    ├── 2b/
    └── ...
```

The notebook will automatically split this into train/val sets.


In [ ]:
# Install PyTorch (if not already installed)
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118


## 1. Mount Google Drive & Check GPU


In [ ]:
# Mount Google Drive (if using)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {gpu_name}")
    print(f"Memory: {gpu_memory:.1f} GB")
else:
    print("No GPU detected! Go to Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU required for training")


## 2. Configuration



In [ ]:
import os
import time
import shutil
import torch
from torch import nn
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms, datasets
from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights
from sklearn.model_selection import train_test_split
from collections import defaultdict

SOURCE_DATA_DIR = "data/segmented"  # Source data with class folders
OUTPUT_DIR = "data/train_val_split"  # Where train/val split will be saved
IMG_SIZE = 600
INITIAL_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4
EPOCHS = 20
LR = 3e-4
CHECKPOINT_DIR = "data/checkpoints/efficientnet_b7/"

# Train/val split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.2

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## 3. Split Dataset into Train/Val


In [ ]:
def split_dataset(source_dir, output_dir, train_ratio=0.8, val_ratio=0.2):
    """Split images from class folders into train/val structure.
    
    This creates new folders at output_dir/train/ and output_dir/val/
    """
    assert abs(train_ratio + val_ratio - 1.0) < 1e-5, "Ratios must sum to 1"

    # Check if output directory already exists and warn
    train_path = os.path.join(output_dir, "train")
    val_path = os.path.join(output_dir, "val")
    
    if os.path.exists(train_path) or os.path.exists(val_path):
        print(f" WARNING: Output directory {output_dir} already contains train/val folders.")

    
    # Create output directories (delete existing if present)
    for split in ['train', 'val']:
        split_path = os.path.join(output_dir, split)
        if os.path.exists(split_path):
            shutil.rmtree(split_path)
            print(f"   Deleted existing {split_path}")
        os.makedirs(split_path)
        print(f"   Created {split_path}")

    # Get class folders
    classes = [d for d in os.listdir(source_dir)
               if os.path.isdir(os.path.join(source_dir, d))]
    classes.sort()
    print(f"Found {len(classes)} classes: {classes}")

    stats = defaultdict(lambda: defaultdict(int))

    for cls in classes:
        cls_path = os.path.join(source_dir, cls)
        images = [f for f in os.listdir(cls_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        if len(images) < 2:
            print(f"⚠️  Warning: Class '{cls}' has only {len(images)} images, skipping")
            continue

        # Split into train and val
        train_imgs, val_imgs = train_test_split(
            images, train_size=train_ratio, random_state=42, shuffle=True
        )

        # Copy images to respective folders
        for split, img_list in [('train', train_imgs), ('val', val_imgs)]:
            split_cls_path = os.path.join(output_dir, split, cls)
            os.makedirs(split_cls_path, exist_ok=True)
            for img in img_list:
                src = os.path.join(cls_path, img)
                dst = os.path.join(split_cls_path, img)
                shutil.copy2(src, dst)
            stats[cls][split] = len(img_list)

    # Print statistics
    print("\n" + "="*50)
    print("Dataset Split Statistics")
    print("="*50)
    print(f"{'Class':<10} {'Train':<10} {'Val':<10} {'Total':<10}")
    print("-" * 50)
    total_train, total_val = 0, 0
    for cls in classes:
        if cls in stats:
            total = stats[cls]['train'] + stats[cls]['val']
            total_train += stats[cls]['train']
            total_val += stats[cls]['val']
            print(f"{cls:<10} {stats[cls]['train']:<10} {stats[cls]['val']:<10} {total:<10}")
    print("-" * 50)
    print(f"{'TOTAL':<10} {total_train:<10} {total_val:<10} {total_train+total_val:<10}")

    return classes

# Verify source data exists
if not os.path.exists(SOURCE_DATA_DIR):
    print(f" Source data directory not found: {SOURCE_DATA_DIR}")
    raise FileNotFoundError(f"Source data directory not found: {SOURCE_DATA_DIR}")

# Run the split
CLASS_NAMES = split_dataset(SOURCE_DATA_DIR, OUTPUT_DIR, TRAIN_RATIO, VAL_RATIO)


## 4. Create Data Loaders


In [ ]:
# Data augmentation for training
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Validation transforms (no augmentation)
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Create datasets from split data
train_path = os.path.join(OUTPUT_DIR, "train")
val_path = os.path.join(OUTPUT_DIR, "val")

train_ds = datasets.ImageFolder(train_path, train_tfms)
val_ds = datasets.ImageFolder(val_path, val_tfms)
num_classes = len(train_ds.classes)

print(f"\n Detected {num_classes} classes: {train_ds.classes}")
print(f"   Training samples: {len(train_ds)}")
print(f"   Validation samples: {len(val_ds)}")


## 5. Model Setup


In [ ]:
# Load pretrained EfficientNet-B7
print("Loading EfficientNet-B7 with ImageNet weights...")
weights = EfficientNet_B7_Weights.IMAGENET1K_V1
model = efficientnet_b7(weights=weights)

# Replace classifier for hair type classification
classifier = model.classifier
in_features = classifier[1].in_features
classifier[0] = nn.Dropout(p=0.5, inplace=True)
classifier[1] = nn.Linear(in_features, num_classes)
model.classifier = classifier

model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n Model loaded successfully")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

# Loss, optimizer, scaler
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = GradScaler()


## 6. Find Optimal Batch Size


In [ ]:
def find_largest_batch_size(initial_bs=INITIAL_BATCH_SIZE):
    """Automatically find the largest batch size that fits in GPU memory."""
    bs = initial_bs
    while bs > 0:
        try:
            test_loader = torch.utils.data.DataLoader(
                train_ds,
                batch_size=bs,
                shuffle=True,
                num_workers=2,
                pin_memory=True
            )
            images, labels = next(iter(test_loader))
            images = images.to(device)
            labels = labels.to(device)
            # Try a forward pass with the model
            with autocast():
                _ = model(images)
            print(f" Batch size {bs} fits in memory.")
            return bs
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f" Batch size {bs} OOM → trying smaller batch...")
                torch.cuda.empty_cache()
                bs //= 2
            else:
                raise e
    raise RuntimeError("Could not find any valid batch size.")

BATCH_SIZE = find_largest_batch_size()
print(f" Using batch size: {BATCH_SIZE}\n")

# Create data loaders with optimal batch size
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True)


## 7. Training Functions


In [ ]:
def validate():
    """Validate the model on validation set."""
    model.eval()
    total, correct = 0, 0
    running_loss = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

    return running_loss / len(val_loader), correct / total


## 8. Training Loop


In [ ]:
best_val_acc = 0.0

print(f"\n{'='*60}")
print("Starting Training")
print(f"{'='*60}")
print(f"Epochs: 1 to {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATION_STEPS})")
print(f"Learning rate: {LR}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"{'='*60}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0
    start = time.time()

    optimizer.zero_grad()

    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)

        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUMULATION_STEPS

    val_loss, val_acc = validate()
    duration = time.time() - start

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Time: {duration:.1f}s")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ckpt_path = os.path.join(CHECKPOINT_DIR, "v7_model.pth")
        torch.save(model.state_dict(), ckpt_path)
        print(f"  New best model saved! (acc: {best_val_acc:.4f})")

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"{'='*60}")


## 9. Save Model for Deployment


In [ ]:
def save_model_for_deployment(model, save_path, num_classes, class_names, img_size):
    """Save model in format compatible with Streamlit app."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'num_classes': num_classes,
        'class_names': list(class_names),
        'img_size': img_size,
        'model_architecture': 'efficientnet_b7'
    }
    torch.save(checkpoint, save_path)
    print(f"Saved deployment model → {save_path}")

# Save the model in the exact format the Streamlit app expects
final_model_path = os.path.join(CHECKPOINT_DIR, "efficientnet_b7_hair_classifier.pth")
save_model_for_deployment(
    model=model,
    save_path=final_model_path,
    num_classes=num_classes,
    class_names=train_ds.classes,
    img_size=IMG_SIZE
)

print(f"\nModel files saved in: {CHECKPOINT_DIR}")
print("Download 'efficientnet_b7_hair_classifier.pth' !")


## 10. Download Model



In [ ]:
from google.colab import files
files.download('data/checkpoints/efficientnet_b7/efficientnet_b7_hair_classifier.pth')
